# ECON 524 — Computational Assignment 4
## Debiased Machine Learning and the Riesz Representer


---

### What this assignment is about
DML has two ideas: (1) an **orthogonal score** makes the target insensitive to first-order errors in ML-estimated nuisances; (2) **cross-fitting** keeps the nuisance learner from grading its own homework. Section 3 of the lecture adds a third: for any target that is a linear functional of a regression, the **Riesz representer** tells you exactly how nuisance errors leak into the target, and **Riesz regression** learns the correction automatically. Q1–Q2 build intuition for (1)–(2); Q3–Q4 are about the Riesz representer.

### Rules
* **Simulated data only.** All data-generating processes (DGPs) are fully specified below. Set seeds so your notebook is reproducible. Monte Carlo sizes ($R$) are **minimums**.
* **Implement the estimators yourself** with `numpy`, `pandas`, `scipy`, `statsmodels`, `scikit-learn` and `matplotlib` (`torch` only for the bonus). Packages such as `doubleml` may be used **only** as an optional cross-check.
* **Every part requires interpretation.** Answer in the markdown cells (typically 4–8 sentences) and refer to slides, formulas, or your tables. Numbers without interpretation earn at most half credit.
* Your notebook must run top to bottom ("Restart & Run All"). Report the runtime of each Monte Carlo.
* Include the **AI-use acknowledgment statement** required by the lab syllabus (README) at the end of the notebook.

**Points:** Q1 25 · Q2 25 · Q3 25 · Q4 25 · Bonus 5.

**Notation (as in the slides).** Outcome $Y$, treatment $D$, controls $Z\in\mathbb R^p$. $\gamma_0(D,Z) = E[Y\mid D,Z]$, $\varrho_0(Z)=P(D=1\mid Z)$, $\alpha_0$ = Riesz representer. For the PLR model: $\ell_0(Z)=E[Y|Z]$, $m_0(Z) = E[D|Z]$. $\Sigma$ denotes the $p\times p$ matrix with $\Sigma_{jk} = 0.5^{|j-k|}$. All error terms are independent $N(0,1)$ unless stated otherwise. "Coverage" means the share of replications in which the nominal 95% interval $\hat\theta\pm1.96\,\widehat{se}$ contains the truth.

In [ ]:
# Setup: imports, seeds, helper functions


---
# Q1. Regularization bias and Neyman orthogonality (25 pts)
*Slides 4–8 and 14–18.*

**DGP 1.** $n=200$, $p=150$, $\theta_0 = 0.5$, $Z\sim N(0,\Sigma)$,
$$D = Z^\top\pi_0 + V,\qquad Y = D\theta_0 + Z^\top\gamma_0 + U,$$
$$\gamma_0 = (1,\,1,\,\underbrace{0.2,\dots,0.2}_{j=3,\dots,7},\,0,\dots,0),\qquad \pi_0 = (0,\,0,\,\underbrace{0.8,\dots,0.8}_{j=3,\dots,7},\,0.5,\,0.5,\,0,\dots,0).$$
Controls $Z_3,\dots,Z_7$ are *weak* predictors of $Y$ but *strong* predictors of $D$: the situation slide 4 warns about.

### Q1(a) [8 pts] Orthogonality, by hand and along a path
1. For the naive score $g_N(X;\theta,\gamma) = (Y-D\theta-Z^\top\gamma)D$, show that $\partial_\gamma E[g_N(X;\theta_0,\gamma)]\big|_{\gamma_0} = -E[DZ]$, and compute this vector for DGP 1 (hint: $E[DZ] = \Sigma\pi_0$). Which coordinates are largest, and what does that imply about the danger of dropping them?
2. For the partialling-out score of slide 16, show that the derivatives with respect to $\beta$ and $\pi$ are both zero. Write out the steps.
3. **Dropping path.** For $t\in[0,1]$, multiply the nuisance coefficients on $Z_3,\dots,Z_7$ by $(1-t)$ ($t=1$: dropped) and solve the *population* moment for $\theta(t)$ in three cases: (N) naive score with $\gamma_t$; (O1) partialling-out score with only $\beta_t$ perturbed ($\beta_0=\gamma_0+\theta_0\pi_0$ is the coefficient of $Y$ on $Z$) and $\pi_0$ correct; (O2) partialling-out score with both $\beta_t$ and $\pi_t$ perturbed. Use closed forms (in terms of $\Sigma$) or one very large simulated sample. Plot $\theta(t)-\theta_0$ for the three cases. Interpret the slope at $t=0$, the value at $t=1$, and what O1 vs O2 says about *why* "double" selection works.

In [ ]:
# Q1(a): your code here


**Q1(a): your answer / interpretation.**

_Write here._

### Q1(b) [9 pts] Monte Carlo: four estimators
With $R\ge 500$ draws from DGP 1, compare:
1. **OLS** of $Y$ on $D$ and all 150 controls (HC1 standard error);
2. **Naive single-selection post-LASSO**: LASSO of $Y$ on $(D,Z)$ keeping $D$ unpenalized, then OLS of $Y$ on $D$ and the selected controls (HC1 se);
3. **Post-double-selection** (Belloni, Chernozhukov & Hansen, 2014): OLS of $Y$ on $D$ and the **union** of the controls selected by a LASSO of $Y$ on $Z$ and a LASSO of $D$ on $Z$ (HC1 se);
4. **Double LASSO / partialling-out** (slides 15–18) with the slide 18 standard error.

Use a plug-in (BCCH) penalty or cross-validation, and justify your choice. Report bias, sd, RMSE, mean se and coverage, plus histograms of $\hat\theta-\theta_0$. Also report how often $Z_3,\dots,Z_7$ are selected by (i) the naive $Y$-equation LASSO and (ii) the $D$-equation LASSO. Interpret: which estimators are orthogonal; why OLS is centered but noisy (recall the $p/n$ discussion of Lab 1); and whether the naive standard error is honest.

In [ ]:
# Q1(b): your code here


**Q1(b): your answer / interpretation.**

_Write here._

### Q1(c) [8 pts] When does orthogonality stop being enough?
Approximate sparsity (slides 17–18) requires the nuisances to be well approximated by $s$ controls with $s^2/n\to0$ (up to logs). Take $n=p=200$, $Z\sim N(0,\Sigma)$, $V,U\sim N(0,1)$, $\theta_0=0.5$, and
$$\gamma_{0j} = \pi_{0j} = c_a\, j^{-a},\qquad a\in\{2,\ 1,\ 0.5\},$$
with $c_a$ chosen so that $\operatorname{Var}(Z^\top\gamma_0) = \operatorname{Var}(Z^\top\pi_0) = 1$ for every $a$ (so only the *shape* changes, not the signal strength). For each $a$ run $R\ge300$ replications of the naive estimator and Double LASSO (PDS optional). Report bias, sd and coverage. Interpret in terms of the product of nuisance errors and the sparsity condition. Name an empirical setting that looks like $a=0.5$.

In [ ]:
# Q1(c): your code here


**Q1(c): your answer / interpretation.**

_Write here._

---
# Q2. Cross-fitting and model-based DML with flexible learners (25 pts)
*Slides 9–12 and 20–25.*

**DGP 2 (partially linear).** $n = 1000$, $p = 30$, $\theta_0 = 1$, $Z\sim N(0,\Sigma)$,
$$m_0(Z) = 0.8\cdot\mathbb 1\{Z_1>0\} + 0.6\,Z_2\mathbb 1\{Z_3>0\} + 0.3\,Z_4,$$
$$g_0(Z) = \mathbb 1\{Z_1>0\} + 0.8\,Z_2\mathbb 1\{Z_3>0\} + 0.5\,Z_4 + 0.5(Z_5^2-1),$$
$D = m_0(Z) + V$, $Y = \theta_0D + g_0(Z) + U$. Note $\ell_0 = \theta_0 m_0 + g_0$. Throughout Q2 use the partialling-out score (slide 22).

### Q2(a) [8 pts] Overfitting bias and cross-fitting
Use DGP 2 with $n=500$. Estimate $\theta_0$ (i) **without** sample splitting (fit both nuisances on the full sample and evaluate the score on the same observations) and (ii) with **5-fold cross-fitting**, for two learner configurations:
* **A:** a fully grown random forest (`min_samples_leaf=1`, `max_features=1.0`) for both $\ell$ and $m$;
* **B:** the same forest for $\ell$, cross-validated LASSO for $m$.

$R\ge40$ per configuration. Report bias, sd, coverage, and a histogram of $t=(\hat\theta-\theta_0)/\widehat{se}$ for configuration B, with and without cross-fitting. Explain why B without cross-fitting fails badly while A without cross-fitting may look acceptable, and why cross-fitting is the right fix in both cases. Your explanation should say how much of $Y_i$'s own noise the in-sample predictions absorb.

In [ ]:
# Q2(a): your code here


**Q2(a): your answer / interpretation.**

_Write here._

### Q2(b) [7 pts] Number of folds and repeated splits
Take **one** draw of DGP 2 (`seed = 2026`, $n=1000$) and the slide 23 forest (`n_estimators=100, max_features=20, max_depth=5, min_samples_leaf=2`) for both nuisances. For $K\in\{2,5,10\}$, repeat cross-fitting with $S=20$ different random splits. Report, for each $K$: the mean and sd of $\hat\theta_s$ across splits, the average $\widehat{se}_s$, and the median-aggregated $\hat\theta^*$ and $\widehat{se}^*$ from slide 12. Plot the $\hat\theta_s$ by $K$. Interpret: what changes with $K$, how large split variability is relative to sampling uncertainty, and why the median (rather than the mean) is used.

In [ ]:
# Q2(b): your code here


**Q2(b): your answer / interpretation.**

_Write here._

### Q2(c) [10 pts] Learners and the product of nuisance errors
1. Show algebraically that, with $\Delta_\ell = \hat\ell-\ell_0$ and $\Delta_m = \hat m - m_0$ (both evaluated out of fold),
$$\hat\theta-\theta_0 = \frac{\mathbb E_n[V_DU] - \mathbb E_n[V_D(\Delta_\ell-\theta_0\Delta_m)] - \mathbb E_n[\Delta_mU] + \mathbb E_n[\Delta_m(\Delta_\ell-\theta_0\Delta_m)]}{\mathbb E_n[\hat V_D^2]}.$$
Which terms have mean zero under cross-fitting, and which is the "product" (second-order) term?
2. Monte Carlo with DGP 2 ($n=1000$, $R\ge 15$ per learner): run 5-fold DML with **(i)** cross-validated LASSO, **(ii)** a random forest, **(iii)** gradient boosting, **(iv)** a neural network (your tuning; justify briefly). For each learner report bias, sd, coverage, average out-of-fold $R^2_Y$ and $R^2_D$, the oracle RMSEs of $\hat\ell$ and $\hat m$ (you know $\ell_0,m_0$), and the average of the product term $\mathbb E_n[\Delta_m(\Delta_\ell-\theta_0\Delta_m)]/\mathbb E_n[\hat V_D^2]$. Plot the bias against the product term across learners.
3. Interpret: what explains the ranking of learners, how does this relate to the $R^2$ table on slide 25, and what would you report in an application?

In [ ]:
# Q2(c): your code here


**Q2(c): your answer / interpretation.**

_Write here._

---
# Q3. The Riesz representer: what it is and why it debiases (25 pts)
*Slides 26–36.*

### Q3(a) [8 pts] Riesz representers you can compute by hand
Let $Z\in\{1,\dots,6\}$ with $P(Z=z) = (0.10, 0.15, 0.20, 0.20, 0.15, 0.20)$ and $\varrho_0(z) = (0.10, 0.30, 0.50, 0.60, 0.80, 0.95)$. A function $\gamma(d,z)$ is a vector in $\mathbb R^{12}$, and $\langle\gamma,\eta\rangle = E[\gamma(D,Z)\eta(D,Z)]$.
1. For each functional below, write $\phi(\gamma) = c^\top\gamma$, compute $\alpha_0$ numerically, **and** derive its closed form by hand in terms of $\varrho_0$, $P(Z=z)$ and $P(D=1)$:
   * ATE: $E[\gamma(1,Z)-\gamma(0,Z)]$;
   * ATT: $E[D\{\gamma(1,Z)-\gamma(0,Z)\}]/P(D=1)$;
   * subgroup effect: $E[\gamma(1,Z)-\gamma(0,Z)\mid Z\ge4]$;
   * policy effect of the targeting rule $\pi(z) = \mathbb 1\{z\ge4\}$ relative to the status quo: $E[\pi(Z)\gamma(1,Z) + (1-\pi(Z))\gamma(0,Z) - \gamma(D,Z)]$.
2. Verify $\phi(\gamma) = E[\alpha_0\gamma]$ for 1,000 random vectors $\gamma$ (report the largest discrepancy).
3. For each functional, which cells receive the largest $|\alpha_0|$, and why does this matter for the precision of any estimator of the target? Relate to overlap.
4. *(Conceptual.)* Now let $Z$ be continuous and the target be $\gamma_0(1,z^*)-\gamma_0(0,z^*)$ at one point $z^*$. Explain why no Riesz representer exists in $L^2(D,Z)$ and what that implies for $\sqrt n$-estimation (slide 29: "$\theta_0$ should depend on $\gamma_0$ continuously").

In [ ]:
# Q3(a): your code here


**Q3(a): your answer / interpretation.**

_Write here._

### Q3(b) [8 pts] Double robustness for the ATT
**DGP 3.** $n=5000$, $p=10$, $Z\sim N(0,I_p)$, $\varrho_0(Z) = \operatorname{expit}\{0.6Z_1 - 0.4Z_2 + 0.4(Z_3^2-1)\}$,
$$Y = Z_1 + 0.5\sin(2Z_2) + 0.5Z_3^2 + D(1 + 0.5Z_1) + \varepsilon .$$
The target is $\theta_{ATT} = E[1+0.5Z_1\mid D=1]$ (compute it by simulation with $10^6$ draws).
1. Starting from $\theta_{ATT} = E[D\{Y-\gamma_0(0,Z)\}]/P(D=1)$, show that the orthogonal score obtained with the Riesz representer of Q3(a) is
$$\psi = \frac{1}{P(D=1)}\Big[D\{Y-\gamma(0,Z)\} - (1-D)\frac{\varrho(Z)}{1-\varrho(Z)}\{Y-\gamma(0,Z)\}\Big],$$
and prove the slide 35 identity $E[\psi(\gamma,\alpha)]-\theta_0 = -E[(\alpha-\alpha_0)(\gamma-\gamma_0)]$ for this case. Note that $\gamma(1,\cdot)$ never appears: why?
2. Run a $2\times2$ design ($R\ge200$): estimate $\gamma(0,\cdot)$ by OLS on the controls using either the **correct** terms $\{1,Z_1,\sin 2Z_2,Z_3^2\}$ or the **wrong** terms $\{1,Z_1,Z_2,Z_3\}$; estimate $\varrho$ by a logit on either the **correct** index terms $\{Z_1,Z_2,Z_3^2\}$ or the **wrong** terms $\{Z_1,Z_2,Z_3\}$. Report the bias of the outcome-regression plug-in, the odds-weighting (IPW) estimator, and the orthogonal (DR) estimator in each cell.
3. In one sample of $n=10^6$ with **both** nuisances wrong, compute the left- and right-hand sides of the identity and show they agree.

In [ ]:
# Q3(b): your code here


**Q3(b): your answer / interpretation.**

_Write here._

### Q3(c) [9 pts] The Riesz representer as an "exchange rate" in high dimensions
**DGP 4.** $n=1000$, $p=100$, $Z\sim N(0,\Sigma)$, $\varrho_0(Z) = \operatorname{expit}\{0.4(Z_1+\dots+Z_5)\}$, and $Y = Z^\top b + D(1+0.5Z_1) + \varepsilon$ with $b_j = 1/j$. The target is the ATE $=1$.
1. Compare ($R\ge100$): **(i)** the plug-in $\mathbb E_n[\hat\gamma(1,Z)-\hat\gamma(0,Z)]$ with $\hat\gamma$ fit by LASSO separately by arm (no correction); **(ii)** IPW $\mathbb E_n[\hat\alpha Y]$ with $\hat\alpha$ = Horvitz–Thompson from an $L_1$-penalized logit; **(iii)** the cross-fitted orthogonal (DR) estimator using both. Report bias, sd and (for ii–iii) coverage.
2. For the plug-in, compute in every replication the **exchange-rate prediction** $\mathbb E_n[\alpha_0(D,Z)\{\hat\gamma(D,Z)-\gamma_0(D,Z)\}]$, using the true $\alpha_0$ and $\gamma_0$. Scatter it against the plug-in error $\hat\theta_{PI}-\theta_0$ and report its average next to the plug-in bias.
3. Interpret: what does the Riesz representer tell you about the error of a plug-in ML estimator, and how does the DR estimator remove it?

In [ ]:
# Q3(c): your code here


**Q3(c): your answer / interpretation.**

_Write here._

---
# Q4. Automatic DML via Riesz regression (25 pts + 5 bonus)
*Slides 37–50.*

**DGP 5 (two-period DiD).** $n=2000$, $p=50$, $Z\sim N(0,I_p)$. Group $D\in\{0,1\}$ (treated vs never treated) with $\varrho_0(Z) = \operatorname{expit}\{0.8Z_1 - 0.5(Z_2^2-1) + 0.5Z_3 - 0.5\}$. Outcomes:
$$Y_{\text{pre}} = Z_1 + Z_4 + 2D + \varepsilon_0,\qquad Y_{\text{post}} = Z_1 + Z_4 + 2D + \text{trend}(Z) + D\,\tau(Z) + \varepsilon_1,$$
$\text{trend}(Z) = Z_1 + 0.5(Z_2^2-1) + 0.75\,Z_1Z_3$, $\tau(Z) = 1 + 0.5Z_1$. The target is $\theta_{ATT} = E[\tau(Z)\mid D=1]$ (compute by simulation with $2\times10^6$ draws). Let $\Delta Y = Y_{\text{post}}-Y_{\text{pre}}$ and $\varphi(z) = \big(1,\ z,\ z_1^2,\dots,z_5^2,\ \{z_iz_j\}_{1\le i<j\le5}\big)$, a dictionary of 66 terms.

### Q4(a) [7 pts] The Riesz loss, and why it balances
1. Show that for any continuous linear functional with representer $\alpha_0$: $\ E[(\alpha-\alpha_0)^2] = E[\alpha^2] - 2E[m(X,\alpha)] + \text{const}$. Write the loss for the linear part of the ATT, $\phi(\gamma) = -E[D\gamma(0,Z)]/P(D=1)$, and compare it with the DiD loss on slide 50.
2. For the sieve $\alpha(d,z) = (1-d)\,\varphi(z)^\top\rho$, derive $\hat\rho$ (with a ridge penalty $\lambda\|\rho\|^2$) and show that, at $\lambda=0$, the first-order condition says: **the controls, weighted by $-\hat\alpha$, reproduce the treated-group mean of every dictionary term**.
3. On one draw of DGP 5, compute $\hat\alpha$, verify the balancing property in a table (a few dictionary terms: treated mean, unweighted control mean, weighted control mean), and compare $\hat\alpha$ with the true $\alpha_0 = -(1-D)\varrho_0/\{P(D=1)(1-\varrho_0)\}$ (scatter and correlation). Why is it useful that you never fitted a propensity score?

In [ ]:
# Q4(a): your code here


**Q4(a): your answer / interpretation.**

_Write here._

### Q4(b) [8 pts] Weak overlap: plug-in, trimming, or Riesz regression?
Use the ATE design of the Nov 13 lab: $Z\sim N(0,I_{20})$, $\varrho_0(Z) = \operatorname{expit}\{\kappa(Z_1+0.5Z_2-0.5Z_3^2+0.5)\}$, $Y = Z_1+\sin(2Z_2)+0.5Z_3^2+0.25Z_4 + D(1+0.5Z_1)+\varepsilon$ (ATE $=1$), $n=2000$, with $\kappa\in\{0.5,1.0,1.5,2.0\}$ and $R\ge200$ per $\kappa$. Use cross-fitted Auto-DML (slide 42) with ridge outcome regressions by arm on $\varphi(z) = (1, z, z_1^2,\dots,z_4^2)$ and three choices of $\hat\alpha$:
1. plug-in Horvitz–Thompson from a logit on $\varphi$;
2. the same with $\hat\varrho$ clipped to $[0.05, 0.95]$ ("trimming");
3. Riesz regression with the linear sieve $[D\varphi,(1-D)\varphi]$ and a ridge penalty (report your $\lambda$).

Report bias, sd, median absolute error, coverage and the median of $\max_i|\hat\alpha_i|$ against $\kappa$, and plot the RMSE against $\kappa$ for the three methods. Interpret: why does Riesz regression degrade gracefully, what does clipping trade off, and is the trimmed estimator still estimating the ATE?

**Bonus (+5).** Add RieszNet (slides 40–41) at $\kappa=1.5$ with $R\ge20$ and discuss.

In [ ]:
# Q4(b): your code here


**Q4(b): your answer / interpretation.**

_Write here._

### Q4(c) [10 pts] Automatic DML for difference-in-differences (slides 49–50)
1. Explain why conditional parallel trends holds in DGP 5 but unconditional parallel trends fails. Map DGP 5 to the slide 49–50 notation ($g$, $t$, $C_{gt}$, $\Delta Y_{gt}$, $\mu_{gt,0}$, $\eta_{g,0}$), write the orthogonal score, and identify its Riesz representer with your result from Q4(a).
2. Monte Carlo ($R\ge200$) comparing: **(i)** the unconditional DiD (difference in mean $\Delta Y$); **(ii)** OLS of $\Delta Y$ on $D$ and $Z$ (linear); **(iii)** the outcome-regression plug-in $\mathbb E_n[D\{\Delta Y-\hat\mu(0,Z)\}]/\hat P(D=1)$ with an ML learner of your choice for $\mu(0,Z)=E[\Delta Y\mid D=0,Z]$, cross-fitted; **(iv)** Auto-DML: the same learner plus the Riesz regression $\hat\alpha$ from Q4(a), cross-fitted, with a standard error from the influence function that accounts for estimating $P(D=1)$. Report bias, sd and (for iv) coverage.
3. Interpret: why does (ii) fail even though it "controls for all of $Z$"; why is (iii) biased; what does (iv) add? Relate to the CS-vs-DML comparison on slides 54–56 and to your Assignment 3.

In [ ]:
# Q4(c): your code here


**Q4(c): your answer / interpretation.**

_Write here._

---
## AI-use acknowledgment
*Paste the acknowledgment statement required by the lab syllabus here, describing which tools you used, for which parts, and how you verified their output.*